---

### Part 1: How the `timeit` Library Works (The Basics)

At its core, `timeit` works like a highly precise stopwatch. Instead of manually recording the start and end times of a single operation (which is highly prone to timing fluctuations), `timeit` does the following:

1. **Looping:** It runs your target function over and over again inside a loop (controlled by the `number` argument). If `number=100`, it runs your code 100 times, records the total time, and divides it to get an average.
2. **Repetition:** To filter out background operating system noise, it repeats this loop several times (controlled by the `repeat` argument).
3. **No Arguments Allowed:** `timeit` expects to run a function that takes **no inputs/arguments**. Because of this, we use Python's **`lambda`** keyword (explained in the code below) to wrap our PyTorch operations so `timeit` can trigger them.

---

### Part 2: Reusable Timing Template for Your Course

This template is designed to be copy-pasted and used across all your future modules. It handles both CPU and GPU operations, performs warmups to prime the engine, synchronizes the hardware, and isolates system noise.

### Why is this code reliable for your course?

* **No String Code:** Standard Python documentation often shows `timeit` using strings of code (e.g., `timeit.repeat("x * 2")`). This makes debugging extremely difficult. The function-based/`lambda` approach used above lets you write normal, syntax-highlighted Python code.
* **Warmups:** By running the warmup loop first, you ensure that initialization overhead doesn't skew your results.
* **Scale-Ready:** If you switch your tensors to the GPU (`device="cuda"`) later in your studies, you can pass `"cuda"` into this function, and it will automatically handle GPU synchronization to keep your measurements accurate [2].

Here is exactly what `timeit` is doing behind the scenes:

---

### Step-by-Step Execution

1. **Repeat 1:**
   * Starts the stopwatch.
   * Runs your code **100 times** in a loop.
   * Stops the stopwatch.
   * Records the total time (e.g., `0.25 seconds`).

2. **Repeat 2:**
   * Starts the stopwatch.
   * Runs your code **100 times** in a loop.
   * Stops the stopwatch.
   * Records the total time (e.g., `0.26 seconds`).

3. **Repeat 3:**
   * Starts the stopwatch.
   * Runs your code **100 times** in a loop.
   * Stops the stopwatch.
   * Records the total time (e.g., `0.35 seconds` — *maybe a background app lagged the CPU here*).

4. **Repeat 4:**
   * Starts the stopwatch.
   * Runs your code **100 times** in a loop.
   * Stops the stopwatch.
   * Records the total time (e.g., `0.24 seconds`).

5. **Repeat 5:**
   * Starts the stopwatch.
   * Runs your code **100 times** in a loop.
   * Stops the stopwatch.
   * Records the total time (e.g., `0.25 seconds`).

At the end, `timeit.repeat()` returns a list of these 5 total times:
`[0.25, 0.26, 0.35, 0.24, 0.25]`

---

### Why is it split up like this?

This design serves two distinct purposes:

* **The `number` argument (loops) is for Timer Accuracy:** Some PyTorch operations take less than a microsecond to run. If you only run them once, the system stopwatch isn't fast or precise enough to measure them accurately. Running them 100 times makes the calculation last long enough for the clock to measure reliably.
* **The `repeat` argument is for System Noise Control:** Running multiple trials allows you to spot outliers (like **Repeat 3** at `0.35 seconds`, which was slowed down by background computer noise). Taking the `min()` of this list ignores that slow outlier and finds the cleanest run (**Repeat 4** at `0.24 seconds`).

In [4]:
import torch
import timeit

In [5]:
def measure_speed(func, device="cpu", warmup_runs=15, repeats=5, loops_per_repeat=100):
    """
    A robust reusable timing utility for PyTorch operations.
    :param func: The function to measure (e.g., lambda: x * 2)
    :param device: "cpu" or "cuda"
    :param warmup_runs: Number of times to run the code to prime the system
    :param repeats: How many times to repeat the loop series (to filter noise)
    :param loops_per_repeat: How many times to run the function inside each loop series
    :return:
    """
    # 1. Warm-up Phase
    # The first few runs in PyTorch are always slow due to engine setup.
    # We run the function a few times first so our measurements are clean.
    for _ in range(warmup_runs):
        func()

    def synchronized_wrapper():
        func()
        if device == "cuda":
            torch.cuda.synchronize()
        elif device == "cpu":
            # If your PyTorch version supports cpu synchronization, use it;
            # otherwise, CPU execution naturally blocks and waits.
            if hasattr(torch, "cpu") and hasattr(torch.cpu, "synchronize"):
                torch.cpu.synchronize()

    # 3. Running the stopwatch
    # timeit.repeat runs our wrapped function 'loops_per_repeat' times,
    # and repeats that entire test 'repeats' times.
    times_list = timeit.repeat(
        synchronized_wrapper,
        repeat=repeats,
        number=loops_per_repeat
    )

    # 4. Filter OS Noise (Take the minimum time)
    # The lowest time represents the run with the least background system lag.
    best_total_time = min(times_list)

    # Calculate how long a single operation took in milliseconds (ms)
    time_per_run_ms = (best_total_time / loops_per_repeat) * 1000

    return time_per_run_ms

In [6]:
# EXAMPLE OF EXECUTION USING THE TIMER
x = torch.randn(10000, 5000, device="cpu")

out_of_place_operation = lambda: x * 2
in_place_operation = lambda: x.mul_(2)

# Measure both operations
oop_ms = measure_speed(out_of_place_operation, device="cpu")
ip_ms = measure_speed(in_place_operation, device="cpu")

print(f"Out_of_place_operation: {oop_ms:.4f} ms per run")
print(f"In_place_operation: {ip_ms:.4f} ms per run")

Out_of_place_operation: 30.4515 ms per run
In_place_operation: 23.6676 ms per run
